# Stock correlation networks: analysis for the article

Reproduces every number and figure in *From Correlation Matrices to Stock Networks*. **All results use real observed prices**; the only synthetic computation is the explicitly labelled planted-block recoverability experiment (cell 7).

**Data are not in the repository.** Regenerate `data/panel_adjusted_close.csv` as described in `data/README.md` (Yahoo Finance adjusted-close files plus the S&P 500 membership history), then run `scripts/audit_universe.py`. Survivorship: 203 of 458 members of 6 Jan 2003 remain in the panel; results describe this survivor panel only.

In [ ]:
import sys, pathlib, json
import pandas as pd
ROOT = pathlib.Path.cwd()
if not (ROOT / 'stocknet').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
R = ROOT / 'results'
assert (ROOT / 'data/panel_adjusted_close.csv').exists(), 'Panel missing: see data/README.md and scripts/audit_universe.py'
pd.Series(json.load(open(ROOT / 'data/derived/panel_summary.json'))).head(12)

## 1. Headline window (252 days ending 2024-12-30): MST vs PMFG vs fixed density
`scripts/run_part1.py` writes `structure_headline.csv` and `regime_table.csv`.

In [ ]:
import runpy; runpy.run_path(str(ROOT/'scripts/run_part1.py'))
print(pd.read_csv(R/'structure_headline.csv').round(3).to_string()); print(pd.read_csv(R/'regime_table.csv').round(3).to_string())

## 2. Four correlation estimators, rolling structure, persistence (several minutes; PMFG is greedy)

In [ ]:
runpy.run_path(str(ROOT/'scripts/run_part2.py'))
for f in ['estimator_pairwise_correlation_agreement','estimator_edge_jaccard_mst','mst_persistence_by_gap']: print(f); print(pd.read_csv(R/f'{f}.csv').round(3).to_string())

## 3. Bootstrap reliability, window length, SYNTHETIC recoverability
Joint moving-block bootstrap (block 21, B=300). The last part is a labelled synthetic mechanism experiment.

In [ ]:
runpy.run_path(str(ROOT/'scripts/run_part3.py'))
print(json.dumps(json.load(open(R/'bootstrap_summary.json')), indent=1)); print(pd.read_csv(R/'synthetic_recoverability.csv').round(3).to_string())

## 4. Walk-forward risk diagnostic (survivor panel, descriptive)
Leakage-free: weights at each month-end use only the previous 252 days. Costs are charged on traded notional.

In [ ]:
runpy.run_path(str(ROOT/'scripts/run_part4.py'))
print(pd.read_csv(R/'walkforward_summary.csv').round(4).to_string())

## 4b. Bootstrap stability versus temporal persistence, and the Steiner-tree experiment
Terminals AAPL, JPM, XOM, JNJ, PG, CAT, HD, SO were fixed before any Steiner computation. Costs are sums of correlation distances; no causal or investment meaning.

In [ ]:
runpy.run_path(str(ROOT/'scripts/run_part5.py'))
print(pd.read_csv(R/'bootstrap_vs_temporal.csv').round(3).T.to_string())
print(pd.read_csv(R/'steiner_primary_terminals.csv')[['window_end','tree','cost','edges','intermediates']].round(3).to_string())

## 5. Figures
Writes Plotly JSON specs to `public/charts/stock-networks/` and static PNG fallbacks to `public/images/writing/stock-networks/` (needs kaleido and a Chromium: set `BROWSER_PATH`).

In [ ]:
import subprocess
_ = subprocess.run([sys.executable, str(ROOT/'scripts/make_figures.py')], check=True)   # writes the Plotly JSON specs and PNG fallbacks